Chapter 1 of [Designing Multi-Agent Systems](https://multiagentbook.com/) compares what a model can answer on its own, what an agent can do with tools, and when multi-agents might be useful. It ends with a small example in which a poet and a critic take turns revising a haiku.

## When a task calls for an agent

Three examples show us what each design layer adds:

- A model can answer a factual question using what it **learned** during training.
- To report today's stock price, an agent can **fetch** (tool call) a quote and use the result in its answer.
- An app that displays prices, buys stock, and files taxes needs work across several domains. Separate agents can be **orchestrated** to take on distinct roles, such as designing the interface and integrating the trading API. The results of testing and integration can change what they need to do next.

## What is an agent?

An **agent** reasons about a task, acts on it, communicates with users or other systems, and adapts to what happens. A stock-price agent might recognize that “today” requires a live quote, call an API, and try another source if that call fails.

- **Model.** An LLM reasons about the task. Program logic or human input can also guide decisions.
- **Tools.** A tool lets the agent do something outside the model, such as run code, drive a browser, or fetch a quote from a particular API.
- **Memory.** (i) Working context holds information needed for the current task. (ii) Longer-term memory can carry information between tasks/sessions. The application or the agent can decide what to keep.

The agent acts, observes the result, and uses it to decide what to do next. Dibia calls the practice of representing these steps as messages *conversational programming* (@fig-understanding-multi-agent-systems-0a9f85207d484f81b60d102e8c68d74b). A tool result can enter the conversation as a message, where the model can use it on the next turn.


![A conversation's tool activity and assistant response. Notice that a lot is happening under the hood from a few user inputs. (Screenshot from Deepseek Harness)](../images/post/understanding-multi-agent-systems-e0bd80262dff43dd9f62d215297cb8ec.png){#fig-understanding-multi-agent-systems-0a9f85207d484f81b60d102e8c68d74b fig-alt="Screenshot of a chat trajectory with model and tool activity bars, bash tool calls, assistant messages, and a user asking why the model is called llm-pi-ai."}

## Why some tasks benefit from several agents

The book points to four sources of difficulty. A task may need a plan because later steps depend on earlier results. It may span several kinds of expertise, require more context than a model can use reliably, or force the system to adapt as new information arrives. A single agent can plan and adapt. The architecture decision focuses on what improves when you divide the work.

Suppose two unrelated tests fail after an upgrade. Each investigator can read one failure log and the relevant code, then report a suspected cause and how to verify it. The investigations can run in parallel, and neither agent needs the other's full transcript. If both failures depend on the same database migration, though, the second diagnosis may depend on the schema produced by the first fix. Passing partial results back and forth can cost more than keeping the investigation in one agent's context.

Specialization can also help when the steps have to run in sequence. **Context isolation** gives each agent its own session with the inputs and tools it needs. An OCR analysis agent can inspect scanned pages and compare the recognized text with the originals. An agent writing unit tests for the parser can work from the parser's interface and a few representative cases, each with sample OCR text and the expected parsed output. Those cases carry the relevant findings into the test-writing session. The scans, raw logs, and analysis transcript that may be irrelevant to the test writer stay in the OCR session.

A controlled study found gains on tasks with parallelizable subtasks and losses on tightly sequential tasks in its benchmarks. It also found that stronger single-agent baselines left less room for gains from coordination. [Kim et al. (2026)](https://www.nature.com/articles/s42256-026-01268-y)


## Two ways to coordinate agents

Orchestration determines how agents share information and who chooses the next step. In a **multi-agent workflow**, the developer defines the routes in advance. A document pipeline might extract text, analyze it, and format the result in a fixed order. In **autonomous orchestration**, a model chooses a route during the run based on what the agents have found so far.

Both designs can run in one process. The book covers agents that communicate across organizational boundaries later, where discovery, permissions, and communication protocols become part of the design.


## Choosing an architecture

Start with the simplest design that has the capabilities the task requires. A model can summarize a document already in its input. To report a live price from a known API, an application can fetch the quote in code and give it to the model. A single agent becomes useful when the application needs to decide what to do with the result, for example whether to try a different source when a quote is stale.

Once there is a reason to divide the work, ask who needs to decide the next step. If the roles and handoffs are known, put them in a workflow. Consider model-driven orchestration when intermediate results can change which specialist should take over or how to divide the remaining work. Each handoff adds a decision and a chance to lose information, so compare it against what one agent can already do.

In the book's Chapter 10 evaluation, direct model calls did well on simple reasoning and used fewer tokens. On the podcast research task, the single agent with search and the multi-agent systems all scored 9.0 out of 10, compared with 3.2 for the model without search. To test whether coordination helps, give the single-agent baseline the same tools and a comparable budget. Track answer quality alongside token use and elapsed time.


## A poet and a critic

The chapter introduces `picoagents`, the Python library it builds from Chapter 4 onward. In this adaptation, one agent writes a haiku about autumn rain and the other suggests revisions. First, we setup the LLM client:

In [6]:
# Adapted from Chapter 1's poet–critic example to use DeepSeek.
# Set DEEPSEEK_API_KEY in the environment or store it in Atlas's vault.
import os

from picoagents import Agent
from picoagents.llm import OpenAIChatCompletionClient
from picoagents.orchestration import RoundRobinOrchestrator
from picoagents.termination import MaxMessageTermination, TextMentionTermination

from atlas.vault import get_secret

MODEL_NAME = "deepseek-flash"
API_KEY = get_secret("DEEPSEEK_API_KEY")
BASE_URL = "https://api.deepseek.com"

client = OpenAIChatCompletionClient(
    model=MODEL_NAME,
    api_key=API_KEY,
    base_url=BASE_URL,
)

Then, we define the `poet` and `critic` agents simply by giving them instructions:

In [7]:
poet = Agent(
    name="poet",
    description="Writes haiku.",
    instructions="You are a haiku poet. Write one haiku per turn.",
    model_client=client,
)

critic = Agent(
    name="critic",
    description="Reviews haiku.",
    instructions=(
        "You review haiku. Give 2-3 specific, actionable suggestions, "
        "or reply with APPROVED alone when no changes are needed."
    ),
    model_client=client,
)

In [15]:
from rich import print_json

poet_response = await poet.run()
print_json(poet_response.model_dump_json())

for message in poet_response.messages:
    print(message.content)

{
  "context": {
    "messages": [
      {
        "content": "Morning dew glistens  \nA small bird sings on the fence  \nSunlight warms the leaves",
        "source": "poet",
        "timestamp": "2026-11-05T01:13:00.395780",
        "role": "assistant",
        "tool_calls": null,
        "structured_content": null,
        "usage": {
          "duration_ms": 2268,
          "llm_calls": 1,
          "tokens_input": 43,
          "tokens_output": 291,
          "tool_calls": 0,
          "memory_operations": 0,
          "cost_estimate": 0.01875
        }
      }
    ],
    "metadata": {},
    "shared_state": {},
    "environment": {},
    "session_id": null,
    "created_at": "2026-11-05T01:10:02.609297",
    "pending_approval_requests": [],
    "approval_responses": {},
    "pending_tool_calls": {}
  },
  "source": "poet",
  "usage": {
    "duration_ms": 2269,
    "llm_calls": 1,
    "tokens_input": 43,
    "tokens_output": 291,
    "tool_calls": 0,
    "memory_operations": 0,
    "cost_estimate": null
  },
  "timestamp": "2026-11-05T01:13:00.395864",
  "finish_reason": "stop"
}

Morning dew glistens  
A small bird sings on the fence  
Sunlight warms the leaves


A `RoundRobinOrchestrator` gives them alternating turns in a shared conversation so the poet can read the critic's feedback before trying again. The critic is instructed to write `APPROVED` once satisfied. The orchestrator checks for that word and limits the number of messages and turns. Chapter 8 returns to the web interface the book uses to display the conversation.

In [ ]:

termination = MaxMessageTermination(max_messages=8) | TextMentionTermination(text="APPROVED")

orchestrator = RoundRobinOrchestrator(
    agents=[poet, critic], 
    termination=termination, 
    max_iterations=4
)

async def main() -> None:
    async for message in orchestrator.run_stream("Write a haiku about autumn rain"):
        print(message)

await main()

[user] 00:28:17 | Write a haiku about autumn rain
[poet] 00:28:19 | Autumn rain whispers—
leaves release their last bright hold
to the wet dark earth.
[critic] 00:28:25 | Two small, actionable revisions could sharpen it:

1. Replace “whispers” with a more concrete rain sound/action (“ticks,” “stings,” “beads on the gutter”) to avoid familiar personification and ground the opening in sensory detail.
2. Make “last bright hold” more concrete—“last gold grip” or “final amber grip”—so the leaf image is visible rather than abstract.
3. Consider putting the turn on “let go” to emphasize motion, e.g.:  
   *Cold autumn rain ticks— / leaves let go their final gold / into dark wet earth.*
[poet] 00:28:28 | Cold autumn rain ticks—
leaves let go their final gold
into dark wet earth.
[critic] 00:28:31 | APPROVED
🏁 RoundRobinOrchestrator: APPROVED | duration: 14.0s, tokens: in:773, out:1.9k, calls: 4 . Stop reason: Composite (any): Text mention found: 'APPROVED'


## What to look for in the conversation

Round-robin fixes who speaks next. The model decides what each agent says on its turn. In the book's sample, the critic gives a suggestion and includes `APPROVED` in the same message. The termination check matches the marker anywhere in the message, so the run ends before the poet can revise the draft. The transcript shows how the system stops. To see the feedback loop in action, look for a run where a specific critique changes the next poem. [PicoAgents termination check](https://github.com/victordibia/designing-multiagent-systems/blob/main/picoagents/src/picoagents/termination/_text_mention.py)

Even if a revision improves the poem, the poet might have made a similar improvement given another turn to review its own work. Compare the poet–critic pair with a poet that reviews and revises its own draft under the same budget. Give both the same prompt, with an explicit 5–7–5 requirement if syllable count is part of the score. Judge the final poems separately from the critic's approval. Check whether each poem follows the requested form and uses concrete imagery that fits the prompt. The transcript can help explain a score by showing which suggestion the poet used and what changed.

Chapter 2 looks at other ways to choose who acts next, including conditional workflows, handoffs, and model-selected turns.
